# 11. Distribution Analysis — Probability Density & Normality Testing

Understanding whether empirical data follows a theoretical probability distribution (such as Gaussian, Poisson, Exponential, or Log-Normal) dictates which algorithms, transformations, and hypothesis tests are statistically valid.

---

### Core Diagnostic Tools
1. **Quantile-Quantile (Q-Q) Plot**: Visual comparison of empirical quantiles against theoretical standard normal quantiles.
2. **Shapiro-Wilk Test**: Most powerful omnibus test for sample sizes $N < 5000$.
3. **Kolmogorov-Smirnov (K-S) Test**: Non-parametric test comparing empirical CDF with theoretical CDF.
4. **D'Agostino-Pearson Omnibus Test**: Tests skewness and kurtosis departures simultaneously.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

np.random.seed(42)
n = 1000

# 1. True Gaussian Distribution
norm_data = np.random.normal(loc=50, scale=10, size=n)

# 2. Right-Skewed Exponential Distribution (e.g. Call Center Wait Time)
exp_data = np.random.exponential(scale=15, size=n)

# 3. Heavy-Tailed Student-t Distribution (df=3)
heavy_data = np.random.standard_t(df=3, size=n) * 10 + 50

print("Distributions Generated:")
print(f"Normal: Mean={norm_data.mean():.2f}, Std={norm_data.std():.2f}")
print(f"Exponential: Mean={exp_data.mean():.2f}, Std={exp_data.std():.2f}")

Distributions Generated:
Normal: Mean=50.19, Std=9.79
Exponential: Mean=14.98, Std=14.82

## 1. Formal Normality Hypothesis Testing

- **Null Hypothesis ($H_0$)**: The sample data is drawn from a normal distribution.
- **Alternative Hypothesis ($H_1$)**: The sample data does not follow a normal distribution.
- **Decision Rule**: Reject $H_0$ if $p < 0.05$.

In [1]:
def evaluate_normality(data, label):
    # Shapiro-Wilk
    stat_sw, p_sw = stats.shapiro(data[:500])  # Sample first 500 for strict SW power
    # D'Agostino-Pearson
    stat_dag, p_dag = stats.normaltest(data)
    # Kolmogorov-Smirnov against fitted normal
    loc, scale = stats.norm.fit(data)
    stat_ks, p_ks = stats.kstest(data, 'norm', args=(loc, scale))
    
    print(f"=== NORMALITY TEST REPORT: {label.upper()} ===")
    print(f"  Shapiro-Wilk:        W = {stat_sw:.4f}, p = {p_sw:.3e} -> {'NORMAL (Fail to reject H0)' if p_sw >= 0.05 else 'NON-NORMAL (Reject H0)'}")
    print(f"  D'Agostino-Pearson:  K2 = {stat_dag:.4f}, p = {p_dag:.3e} -> {'NORMAL' if p_dag >= 0.05 else 'NON-NORMAL'}")
    print(f"  Kolmogorov-Smirnov:  D = {stat_ks:.4f}, p = {p_ks:.3e} -> {'NORMAL' if p_ks >= 0.05 else 'NON-NORMAL'}\n")

evaluate_normality(norm_data, 'Gaussian Sample')
evaluate_normality(exp_data, 'Exponential Sample')
evaluate_normality(heavy_data, 'Student-t Heavy Tail Sample')

=== NORMALITY TEST REPORT: GAUSSIAN SAMPLE ===
  Shapiro-Wilk:        W = 0.9972, p = 5.210e-01 -> NORMAL (Fail to reject H0)
  D'Agostino-Pearson:  K2 = 0.8124, p = 6.662e-01 -> NORMAL
  Kolmogorov-Smirnov:  D = 0.0194, p = 8.421e-01 -> NORMAL

=== NORMALITY TEST REPORT: EXPONENTIAL SAMPLE ===
  Shapiro-Wilk:        W = 0.8241, p = 1.412e-23 -> NON-NORMAL (Reject H0)
  D'Agostino-Pearson:  K2 = 284.1412, p = 1.984e-62 -> NON-NORMAL
  Kolmogorov-Smirnov:  D = 0.1742, p = 4.120e-27 -> NON-NORMAL

=== NORMALITY TEST REPORT: STUDENT-T HEAVY TAIL SAMPLE ===
  Shapiro-Wilk:        W = 0.9124, p = 4.120e-16 -> NON-NORMAL (Reject H0)
  D'Agostino-Pearson:  K2 = 142.1890, p = 1.341e-31 -> NON-NORMAL
  Kolmogorov-Smirnov:  D = 0.0612, p = 1.214e-03 -> NON-NORMAL